# Week 3: Recommendation Systems

<small>OPAN 6604. Dataset: MovieLens (`MovieLense-Movies.csv`, `MovieLense-Ratings.csv`) - 610 users, ~9,700 movies, ~100K ratings. Goal: predict the rating a user would give a movie they haven't seen, then turn those predictions into top-N recommendations.</small>

<small>This is the in-class demo: **collaborative filtering** with the [`surprise`](http://surpriselib.com/) library, user-based CF (UBCF), item-based CF (IBCF), and a proper hold-out evaluation. </small>

<small>**Note on similarity:** both CF flavors rest on a similarity measure between rows (users) or columns (movies) of the rating matrix. We use cosine and Pearson - the two covered in lecture.</small>

### Step 0: Setup

<small>Install `scikit-surprise` if needed (pick the path for your environment in the cell below), then import the CF building blocks.</small>

In [37]:
# Install scikit-surprise if it's not already available.
#   All platforms (Colab / Windows / macOS / Linux):
# Note: `numpy<2.0` is required because scikit-surprise's compiled extensions
#   were built against the NumPy 1.x and break under NumPy 2.x.

#       %pip install "numpy<2.0"
#       %pip install scikit-surprise

#   Fallback (only if pip can't find a wheel for your Python and tries to
#   compile, e.g. "Microsoft Visual C++ required") -> use conda-forge:
#       conda install -c conda-forge scikit-surprise -y


In [38]:
# Surprise's CF building blocks:
#   KNNBasic        — neighborhood CF (UBCF or IBCF, toggled via sim_options)
#   BaselineOnly    — global/user/item-mean baseline, useful for comparison
#   Dataset/Reader  — wrap a pandas DataFrame as a Surprise dataset
#   accuracy        — RMSE, MAE, etc. on prediction lists
import pandas as pd
import numpy as np
from collections import defaultdict
from surprise import KNNBasic, BaselineOnly, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

### Step 1: Load data

<small>Load the catalog and the ratings, and build a `movieId` → title lookup for displaying recommendations later. Each `movieId` is unique, so we key everything off it.</small>

In [39]:
# Load the movie catalog and the ratings. movieId is the unique key we use
# throughout; title is only for display, so we build a movieId -> title lookup.
movies = pd.read_csv("MovieLense-Movies.csv")
ratings = pd.read_csv("MovieLense-Ratings.csv")
title_of = dict(zip(movies["movieId"], movies["title"]))

print(f"Movies: {movies.shape}  |  Ratings: {ratings.shape}")

Movies: (9742, 3)  |  Ratings: (100836, 3)


### Step 2: Build the Surprise dataset

<small>Wrap the long-format ratings into a `surprise` dataset. `Reader` declares the rating scale. `build_full_trainset()` uses every rating for training. We'll implement a proper train/test split in Part 2.</small>

In [40]:
# Wrap the long-format ratings into a Surprise dataset.
# MovieLens uses a 0.5–5.0 half-star scale. Using Reader we clip predictions to this range.
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)

# build_full_trainset() uses every rating for training - useful for the
# illustrative top-N below; we'll redo this with a proper hold-out split
# in the evaluation section.
trainset = data.build_full_trainset()
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} movies, {trainset.n_ratings} ratings")

Trainset: 610 users, 9724 movies, 100836 ratings


---
## Part 1: Collaborative Filtering

<small>Collaborative filtering predicts a user's rating for an unseen item from patterns in the rating matrix alone - no item features required. Two perspectives: **user-based** (find users who rate like you) and **item-based** (find items rated like the ones you liked). Both use `KNNBasic`, toggled via `sim_options`.</small>

### User-Based CF

<small>Similarity is computed between **users** (rows): each prediction averages the `k` most similar users who rated the target movie - "people like you also enjoyed...". We use **Pearson**, which centers each user on their own mean - important because users rate on different scales (some skew high, some low).</small>

In [41]:
# User-based: the k=10 most similar users (Pearson similarity) vote on each prediction.
ubcf = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True}, verbose=False)
ubcf.fit(trainset)

### Top-N Recommendations

<small>CF tends to surface rarely-rated movies. Restricting candidates to movies with at least `MIN_RATINGS` ratings is a simple, effective fix. The helper below is reused for both models.</small>

In [42]:
# Recommend a user's top-N unseen movies, ranked by predicted rating.
# Popularity filter (>= MIN_RATINGS): drop movies with very few ratings, which CF
# could otherwise rank highly on almost no evidence (e.g. a single 5/5 rating).
# MIN_RATINGS=20 is just a suggested value - the right cutoff depends on the
# dataset size, catalog, and use case.
MIN_RATINGS = 20
counts = ratings["movieId"].value_counts()
popular_movies = set(counts[counts >= MIN_RATINGS].index)

def top_n_for_user(model, user_id, top_n=5):
    seen = set(ratings.loc[ratings["userId"] == user_id, "movieId"])
    # score every unseen, popular movie; keep the top_n highest predictions
    scored = [(title_of[m], model.predict(user_id, m).est)
              for m in movies["movieId"]
              if m not in seen and m in popular_movies]
    return sorted(scored, key=lambda x: -x[1])[:top_n]

top_n_for_user(ubcf, user_id=1)   # top-5 UBCF recommendations for user 1

[('Departed, The (2006)', 4.783577729244594),
 ('Harold and Maude (1971)', 4.722233912284893),
 ('Postman, The (Postino, Il) (1994)', 4.698747851370958),
 ('Bridge on the River Kwai, The (1957)', 4.685767808716011),
 ('Shawshank Redemption, The (1994)', 4.65890061720083)]

### Item-Based CF

<small>Now similarity is between **movies** (columns), from how the same users rated them - "because you liked X, you might like Y...". We also switch to **cosine**, the common metric for item-based CF. So *both* knobs change from UBCF - the axis (users→movies) and the similarity - which mirrors how the metric is usually chosen per method in practice. Compare this top-5 to the UBCF list: same user, different lens, different movies.</small>

In [43]:
# Item-based: similarity between movies, using cosine (the common item-based choice).
ibcf = KNNBasic(k=10, sim_options={"name": "cosine", "user_based": False}, verbose=False)
ibcf.fit(trainset)

In [44]:
# Top-5 IBCF recommendations for user 1 (same helper, different model).
top_n_for_user(ibcf, user_id=1)

[('Little Shop of Horrors (1986)', 5.0),
 ('Tropic Thunder (2008)', 4.9),
 ('Gone Girl (2014)', 4.9),
 ('Ex Machina (2015)', 4.9),
 ('Amadeus (1984)', 4.800109253267214)]

---
## Part 2: Evaluation

<small>CF is trained without labeled "correct answers," yet we evaluate it like supervised learning: hold out known ratings, predict them, and compare. We report a **rating-accuracy** metric (RMSE) and **ranking** metrics (Precision@K, Recall@K) - the two families from lecture.</small>

### Precision and Recall @ K

<small>Frame recommendation as classification: a movie is "relevant" if its true rating ≥ a threshold (4.0 here). For each user we rank predictions by estimated rating, then ask — of the top-N we'd show, how many were relevant (**precision**), and of all their relevant movies, how many made the top-N (**recall**). We average across users.</small>

In [45]:
# Top-N precision and recall for ranking quality:
#   - a movie is "relevant" if its true rating is >= threshold (4.0 here),
#   - for each user we rank predictions by estimated rating,
#   - precision = fraction of the user's top-N that were actually relevant,
#   - recall    = fraction of the user's relevant items captured in the top-N.
# `top_n` is the list length - distinct from the model's k (the neighborhood size).
# Returns the mean across all users in the predictions list.
def precision_recall_at_k(predictions, top_n=10, threshold=4.0):
    user_data = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_data[uid].append((est, true_r))
    precisions, recalls = [], []
    for items in user_data.values():
        items.sort(key=lambda x: x[0], reverse=True)
        n_relevant = sum(1 for _, t in items if t >= threshold)
        n_hits = sum(1 for _, t in items[:top_n] if t >= threshold)
        precisions.append(n_hits / top_n)
        if n_relevant > 0:
            recalls.append(n_hits / n_relevant)
    return np.mean(precisions), np.mean(recalls)

### Train/Test Split

<small>Replace the full trainset with a 90/10 hold-out so predictions can be scored against ratings the model never saw. `random_state` keeps the split reproducible.</small>

In [46]:
# Replace the full trainset with a 90/10 train/test split so we can score
# predictions against held-out ratings. `random_state` makes the split
# reproducible across runs.
trainset, testset = train_test_split(data, test_size=0.1, random_state=6604)
print(f"Train: {trainset.n_ratings} ratings  |  Test: {len(testset)} ratings")

Train: 90752 ratings  |  Test: 10084 ratings


### Model Comparison

<small>Fit a simple **baseline** (global + per-user + per-item means) alongside the two Part 1 configurations - UBCF (Pearson) and IBCF (cosine) - on the 90% training split, and score them on the held-out 10%. The baseline is the bar CF must clear. Note: a model's rating error (RMSE) and its ranking quality (Precision@10) don't always agree.</small>

In [47]:
# In practice you wouldn't hand-pick k and the similarity metric - you'd tune them with
# cross-validation (surprise.model_selection.GridSearchCV / RandomizedSearchCV), scoring on
# the metric you actually care about: for a top-N list that's a ranking metric (Precision@K
# or NDCG@K), not RMSE. Production systems also use time-based splits and confirm the winner
# with an online A/B test.
models = {
    "Baseline": BaselineOnly(verbose=False),
    "UBCF pearson": KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True}, verbose=False),
    "IBCF cosine": KNNBasic(k=10, sim_options={"name": "cosine", "user_based": False}, verbose=False),
}

results = []
for name, m in models.items():
    m.fit(trainset)
    preds = m.test(testset)
    p, r = precision_recall_at_k(preds, top_n=10, threshold=4.0)
    results.append({
        "Model": name,
        "RMSE": accuracy.rmse(preds, verbose=False),
        "Precision@10": p,
        "Recall@10": r,
    })

pd.DataFrame(results).round(4)

,Model,RMSE,Precision@10,Recall@10
0,Baseline,0.8569,0.4220,0.8171
1,UBCF pearson,0.9775,0.4207,0.8149
2,IBCF cosine,1.0131,0.3566,0.7618
